# Rule validation — IMF baseline + rule-layer tuning

Data is already split into **validation / test** (separate parquet, no more random splitting):
- **validation**: fraud `2024-10` + `normal_2024_10`  -> tune threshold
- **test**: fraud `2024-11` ~ `2025-10` + `normal_2024_11`  -> held-out validation

**Method**:
- **baseline** = `option5_reservoir_sampled_imf_anomaly_score` (0-1 model score), take top-k descending, `k = base_fraud_rate`
- **rule layer** = on the people baseline missed, find `(col, threshold)` among the 24 rule score columns (`TYPE_METRIC`)
- two objectives: **A preserve precision** (P >= floor, max recall) / **B preserve recall** (R >= floor, max precision)
- plot: baseline net-size curve + **Extra rule-based** recall line

> All `%%pyspark`, sharing the same spark namespace. Tune on validation; for final validation change `df_eval` to `test` and rerun.


In [22]:
%%pyspark project.spark.compatibility
from pyspark.sql import functions as F
from pyspark.sql import SparkSession
import pandas as pd
import numpy as np
import boto3

spark = (
    SparkSession.builder
    .appName("rule_validation_combine")
    .config("spark.executor.memory", "18g")
    .config("spark.driver.memory", "10g")
    .config("spark.executor.cores", "4")
    .config("spark.sql.shuffle.partitions", "300")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
    .getOrCreate()
)



Connection: project.spark.compatibility | Run start time: 2026-06-09 17:40:50.166131 | Run duration : 0:00:02.329440s.


In [23]:
%%pyspark project.spark.compatibility
validation = spark.read.parquet('s3://enstream-lake-gold-dev/poc/analysis/rule_based/v2/validation_final/')
test       = spark.read.parquet('s3://enstream-lake-gold-dev/poc/analysis/rule_based/v2/test_final/')
print(f"validation: {validation.count():,}")
print(f"test:       {test.count():,}")


validation: 1,001,588
test:       1,016,384

Connection: project.spark.compatibility | Run start time: 2026-06-09 17:40:52.498815 | Run duration : 0:00:07.717793s.


In [24]:
%%pyspark project.spark.compatibility
import math

TYPES   = ["FPF", "TrueName", "Synthetic", "Mule", "ATO", "Combined"]
METRICS = ["hits", "score", "hits_with_uncertain", "score_with_uncertain"]
RAW_COL   = "option5_reservoir_sampled_imf_raw_score"
PRED_COL  = "option5_reservoir_sampled_imf_anomaly_score"
LABEL_COL = "true_label"

# binary true_label: anything that is not non_fraud is fraud -- added for both validation and test
def add_label(df):
    return df.withColumn(LABEL_COL,
        F.when(F.col("fraud_type") == "non_fraud", F.lit("non_fraud")).otherwise(F.lit("fraud")))

validation = add_label(validation)
test       = add_label(test)

df_eval = validation        # tune on validation; for final validation change this line (and the pdf cell below) to test



Connection: project.spark.compatibility | Run start time: 2026-06-09 17:41:00.220198 | Run duration : 0:00:02.416912s.


In [27]:
%%pyspark project.spark.compatibility
selected_cols = [
    "true_label",

    "FPF_hits",
    "FPF_score",
    "FPF_hits_with_uncertain",
    "FPF_score_with_uncertain",

    "TrueName_hits",
    "TrueName_score",
    "TrueName_hits_with_uncertain",
    "TrueName_score_with_uncertain",

    "Synthetic_hits",
    "Synthetic_score",
    "Synthetic_hits_with_uncertain",
    "Synthetic_score_with_uncertain",

    "Mule_hits",
    "Mule_score",
    "Mule_hits_with_uncertain",
    "Mule_score_with_uncertain",

    "ATO_hits",
    "ATO_score",
    "ATO_hits_with_uncertain",
    "ATO_score_with_uncertain",

    "Combined_hits",
    "Combined_score",
    "Combined_hits_with_uncertain",
    "Combined_score_with_uncertain",

    "option5_reservoir_sampled_imf_raw_score",
    "option5_reservoir_sampled_imf_anomaly_score",
]

test_selected = test.select(*selected_cols)
validation_selectd = validation.select(*selected_cols)


Connection: project.spark.compatibility | Run start time: 2026-06-09 17:44:07.939983 | Run duration : 0:00:02.287852s.


In [31]:
%%pyspark project.spark.compatibility
def fn(df):
    d = {r[LABEL_COL]: r["count"] for r in df.groupBy(LABEL_COL).count().collect()}
    return d.get("non_fraud", 0), d.get("fraud", 0)

v_norm, v_fraud = fn(validation)
t_norm, t_fraud = fn(test)
print(f"validation: normal={v_norm:,}, fraud={v_fraud:,}")
print(f"test:       normal={t_norm:,}, fraud={t_fraud:,}")
print(f"total:      normal={v_norm + t_norm:,}, fraud={v_fraud + t_fraud:,}")

# baseline_k = fraud rate of validation
fraud = v_fraud
normal = v_norm
# BASELINE_K = 0.02
BASELINE_K = fraud / (fraud + normal)
print(f"BASELINE_K = {BASELINE_K:.6f}")


validation: normal=1,000,000, fraud=1,588
test:       normal=1,000,000, fraud=16,384
total:      normal=2,000,000, fraud=17,972
BASELINE_K = 0.001585

Connection: project.spark.compatibility | Run start time: 2026-06-09 17:45:58.362440 | Run duration : 0:00:06.223997s.


In [32]:
%%pyspark project.spark.compatibility
SCORE_COLS = [f"{t}_{m}" for t in TYPES for m in METRICS if f"{t}_{m}" in df_eval.columns]
pdf = df_eval.select(LABEL_COL, PRED_COL, *SCORE_COLS).toPandas()
lab         = (pdf[LABEL_COL] == "fraud").astype(int).values
N           = len(pdf)
total_fraud = int((lab == 1).sum())
total_norm  = int((lab == 0).sum())
print(f"rows={N:,} | fraud={total_fraud:,} | normal={total_norm:,} | score cols={len(SCORE_COLS)}")


rows=1,001,588 | fraud=1,588 | normal=1,000,000 | score cols=24

Connection: project.spark.compatibility | Run start time: 2026-06-09 17:46:04.589687 | Run duration : 0:00:18.997271s.


In [33]:
%%pyspark project.spark.compatibility
# baseline: PRED_COL descending top-k (exactly k_cnt rows, method='first')
k_cnt    = math.ceil(N * BASELINE_K)
rank_raw = pd.Series(pdf[PRED_COL].values).rank(method="first", ascending=False).values
base_hit = rank_raw <= k_cnt
base_f   = int((base_hit & (lab == 1)).sum())
base_n   = int((base_hit & (lab == 0)).sum())
print("hits =", int(base_hit.sum()), "| k_cnt =", k_cnt)
print(f"caught fraud={base_f:,}, flagged normal={base_n:,}, sum={base_f + base_n:,}")


命中数 = 1588 | k_cnt = 1588
caught fraud=1,103, flagged normal=485, sum=1,588

Connection: project.spark.compatibility | Run start time: 2026-06-09 17:46:52.231651 | Run duration : 0:00:04.946137s.


In [34]:
%%pyspark project.spark.compatibility
P_base  = base_f / max(base_f + base_n, 1)
R_base  = base_f / max(total_fraud, 1)
F1_base = 2 * P_base * R_base / max(P_base + R_base, 1e-9)
print(f"baseline:  P={P_base:.4f}  R={R_base:.4f}  F1={F1_base:.4f}  (caught={base_f:,}, fp={base_n:,})")


baseline:  P=0.6946  R=0.6946  F1=0.6946  (caught=1,103, fp=485)

Connection: project.spark.compatibility | Run start time: 2026-06-09 17:48:29.281437 | Run duration : 0:00:04.696470s.


In [36]:
%%pyspark project.spark.compatibility
import numpy as np, pandas as pd

missed = (~base_hit) & (lab == 1)              # fraud missed by baseline
ms = pd.Series(pdf[PRED_COL].values[missed])

print(f"fraud missed by baseline: {int(missed.sum())}\n")
print(f"their anomaly scores:  min={ms.min():.4f}  median={ms.median():.4f}  "
      f"mean={ms.mean():.4f}  max={ms.max():.4f}")
print(f"  of which =0: {int((ms == 0).sum())},  >0.5: {int((ms > 0.5).sum())}\n")
print("counts bucketed by 0.05:")
binned = (np.floor(ms / 0.05) * 0.05).round(2)
print(binned.value_counts().sort_index().to_string())


baseline 漏掉的 fraud: 485 个

它们的 anomaly 分:  min=0.0000  median=0.2516  mean=0.2166  max=0.4984
  其中 =0 的 162 个，  >0.5 的 0 个

按 0.05 分桶计数:
0.00    166
0.05      5
0.10      7
0.15      9
0.20     55
0.25     95
0.30     25
0.35      6
0.40     53
0.45     64

Connection: project.spark.compatibility | Run start time: 2026-06-09 17:52:59.126581 | Run duration : 0:00:04.755204s.


In [43]:
%%pyspark project.spark.compatibility
import numpy as np, pandas as pd
RULE = "Combined_score_with_uncertain"

# (1) rule scores of the 485 missed low-anomaly fraud
missed = (~base_hit) & (lab == 1)
rs = pd.Series(pdf[RULE].values[missed])
print(f"rule scores of the {int(missed.sum())} missed fraud (anomaly <= 0.5):")
print(f"  =0: {int((rs == 0).sum())} | >0: {int((rs > 0).sum())} | max={rs.max():.1f}")
print(f"  distribution: {rs.round(1).value_counts().sort_index().to_dict()}\n")

# (2) all fraud: high anomaly (caught by IMF) vs low anomaly (missed by IMF), how many of each carry rule signal
fr = lab == 1
d = pd.DataFrame({"anomaly": pdf[PRED_COL].values[fr], "rule": pdf[RULE].values[fr]})
d["group"] = np.where(d.anomaly > 0.5, "high anomaly>0.5 (IMF caught)", "low anomaly<=0.5 (IMF missed)")
print(d.groupby("group").agg(
    fraud_count=("rule", "size"),
    has_signal_frac=("rule", lambda x: round((x > 0).mean(), 3)),
    rule_median=("rule", "median"),
    rule_max=("rule", "max"),
).to_string())


漏掉的 fraud 485 个（anomaly ≤ 0.5）的规则分:
  =0 的 485 个 | >0 的 0 个 | max=0.0
  分布: {0.0: 485}

                      fraud数  有规则信号比例  规则分_中位  规则分_max
组                                                     
低anomaly≤0.5 (IMF漏掉)     485     0.00     0.0      0.0
高anomaly>0.5 (IMF抓到)    1103     0.18     0.0     15.0

Connection: project.spark.compatibility | Run start time: 2026-06-09 18:06:35.939136 | Run duration : 0:00:04.648599s.


In [37]:
%%pyspark project.spark.compatibility
import numpy as np, pandas as pd

# anomaly of all normal
norm_all = pd.Series(pdf[PRED_COL].values[lab == 0])
print(f"all normal: {len(norm_all):,}")
print(f"  anomaly: min={norm_all.min():.4f}  max={norm_all.max():.4f}  distinct={norm_all.nunique()}")
print(f"  value distribution: {norm_all.round(4).value_counts().to_dict()}")

# the 485 normal pulled into baseline's top-1588  vs  those not pulled in
flagged   = pd.Series(pdf[PRED_COL].values[( base_hit) & (lab == 0)])
unflagged = pd.Series(pdf[PRED_COL].values[(~base_hit) & (lab == 0)])
print(f"\nnormal pulled into top-1588: {len(flagged):,} -> anomaly values {flagged.unique()}")
print(f"normal not pulled in:         {len(unflagged):,} -> anomaly values {unflagged.unique()}")


全部 normal: 1,000,000 个
  anomaly: min=0.5000  max=0.5000  distinct=1
  取值分布: {0.5: 1000000}

被圈进 top-1588 的 normal: 485 个 → anomaly 取值 [0.5]
没被圈的 normal:           999,515 个 → anomaly 取值 [0.5]

Connection: project.spark.compatibility | Run start time: 2026-06-09 17:53:57.424205 | Run duration : 0:00:04.713182s.


In [39]:
%%pyspark project.spark.compatibility
# A) preserve precision: combined P stays above the floor, maximize recall
MIN_PRECISION = 0.6          # knob: lower it to trade for more recall
rp = rule_search("precision", MIN_PRECISION)
best_p = show(rp, "comb_R", f"=== A) preserve precision  P>={MIN_PRECISION:.4f}, pick max recall (sorted by comb_R) ===",
              "(no column can catch more fraud at this precision -> lower MIN_PRECISION)")



=== A) 保精度  P>=0.6000，选 recall 最大（按 comb_R 排）===
                          rule  threshold  extra_fraud  extra_normal  comb_P  comb_R      dP  dR  comb_F1
                      FPF_hits       22.0            0             1  0.6941  0.6946 -0.0004 0.0   0.6944
                     FPF_score       60.0            0             1  0.6941  0.6946 -0.0004 0.0   0.6944
       FPF_hits_with_uncertain       26.0            0             2  0.6937  0.6946 -0.0009 0.0   0.6941
      FPF_score_with_uncertain       65.5            0             1  0.6941  0.6946 -0.0004 0.0   0.6944
                 TrueName_hits        5.0            0             4  0.6928  0.6946 -0.0017 0.0   0.6937
                TrueName_score       13.0            0             4  0.6928  0.6946 -0.0017 0.0   0.6937
  TrueName_hits_with_uncertain        6.0            0             5  0.6924  0.6946 -0.0022 0.0   0.6935
 TrueName_score_with_uncertain       14.0            0             4  0.6928  0.6946 -0.0017 0.0   0.6

In [35]:
%%pyspark project.spark.compatibility
# B) preserve recall: combined R not below the floor, maximize precision
MIN_RECALL = 0.6               # knob: must be > R_base to be meaningful
rr = rule_search("recall", MIN_RECALL)
best_r = show(rr, "comb_P", f"=== B) preserve recall  R>={MIN_RECALL:.4f}, pick max precision (sorted by comb_P) ===",
              "(no column can reach this recall floor while keeping P>0 -> lower MIN_RECALL)")



=== B) 保召回  R>=0.6000，选 precision 最高（按 comb_P 排）===
                          rule  threshold  extra_fraud  extra_normal  comb_P  comb_R      dP  dR  comb_F1
                      FPF_hits       22.0            0             1  0.6941  0.6946 -0.0004 0.0   0.6944
                     FPF_score       60.0            0             1  0.6941  0.6946 -0.0004 0.0   0.6944
      FPF_score_with_uncertain       65.5            0             1  0.6941  0.6946 -0.0004 0.0   0.6944
 Combined_score_with_uncertain       83.5            0             1  0.6941  0.6946 -0.0004 0.0   0.6944
  Combined_hits_with_uncertain       37.0            0             1  0.6941  0.6946 -0.0004 0.0   0.6944
       ATO_hits_with_uncertain       24.0            0             1  0.6941  0.6946 -0.0004 0.0   0.6944
     Mule_score_with_uncertain       51.5            0             1  0.6941  0.6946 -0.0004 0.0   0.6944
      Mule_hits_with_uncertain       26.0            0             1  0.6941  0.6946 -0.0004 0.0   

In [38]:
%%pyspark project.spark.compatibility
miss = ~base_hit

def rule_search(objective, floor):
    """objective='precision': guarantee combined P>=floor, pick max recall;
       objective='recall'   : guarantee combined R>=floor, pick max precision.
       fires only on the people baseline missed (miss), combined = baseline OR (col>=thr)."""
    rows = []
    for col in SCORE_COLS:
        sv = pdf[col].values[miss].astype(float); lv = lab[miss]
        order = np.argsort(-sv, kind="mergesort")
        sv_s, lv_s = sv[order], lv[order]
        cum_f = np.cumsum(lv_s == 1); cum_n = np.cumsum(lv_s == 0)
        tp = base_f + cum_f; fp = base_n + cum_n
        R = tp / max(total_fraud, 1); P = tp / np.maximum(tp + fp, 1)
        if objective == "precision":
            feas = (sv_s > 0) & (P >= floor)
            if not feas.any(): continue
            ii = np.where(feas)[0]; m = ii[int(np.argmax(R[ii]))]
        else:
            feas = (sv_s > 0) & (R >= floor)
            if not feas.any(): continue
            ii = np.where(feas)[0]; m = ii[int(np.argmax(P[ii]))]
        thr = float(sv_s[m])
        fire = miss & (pdf[col].values >= thr) & (pdf[col].values > 0)
        ef = int((fire & (lab == 1)).sum()); en = int((fire & (lab == 0)).sum())
        tpf, fpf = base_f + ef, base_n + en
        Pf, Rf = tpf / max(tpf + fpf, 1), tpf / max(total_fraud, 1)
        rows.append(dict(rule=col, threshold=round(thr, 4), extra_fraud=ef, extra_normal=en,
                         comb_P=round(Pf, 4), comb_R=round(Rf, 4),
                         dP=round(Pf - P_base, 4), dR=round(Rf - R_base, 4),
                         comb_F1=round(2*Pf*Rf/max(Pf+Rf, 1e-9), 4)))
    return pd.DataFrame(rows)

def show(df, sort_col, title, empty_msg):
    print(); print(title)
    if df.empty:
        print(empty_msg); return None
    df = df.sort_values(sort_col, ascending=False).reset_index(drop=True)
    print(df.to_string(index=False)); b = df.iloc[0]
    print(f">>> best: {b['rule']} >= {b['threshold']} | "
          f"P {P_base:.3f}->{b['comb_P']:.3f} ({b['dP']:+.3f}) | "
          f"R {R_base:.3f}->{b['comb_R']:.3f} ({b['dR']:+.3f}) | +{b['extra_fraud']}f / +{b['extra_normal']}n")
    return b



Connection: project.spark.compatibility | Run start time: 2026-06-09 17:55:15.058618 | Run duration : 0:00:02.299014s.


In [ ]:
%%pyspark project.spark.compatibility



In [40]:
%%pyspark project.spark.compatibility
import numpy as np, pandas as pd, math

RULE_COL   = "Combined_score_with_uncertain"
BASELINE_K = 0.02                       # baseline net size (k_cnt ~= 20328 > 16384, recall not blocked by the arithmetic wall)

p    = test.select("fraud_type", PRED_COL, RULE_COL).toPandas()
y    = (p["fraud_type"] != "non_fraud").astype(int).values
pred = p[PRED_COL].astype(float).values
rule = p[RULE_COL].astype(float).values
F, N = max(int((y == 1).sum()), 1), len(y)
rank = pd.Series(pred).rank(method="first", ascending=False).values
base = rank <= math.ceil(N * BASELINE_K)

def pr(mask):
    tp = int((mask & (y == 1)).sum()); fp = int((mask & (y == 0)).sum())
    return tp / max(tp + fp, 1), tp / F

bP, bR = pr(base)
print(f"TEST | total_fraud={F:,} | baseline k={BASELINE_K} (caught {int(base.sum()):,} people)")
print(f"baseline before:  recall={bR:.4f}   precision={bP:.4f}\n")
print(f"baseline OR ({RULE_COL} >= threshold):")
rows = []
for thr in [83.5, 50, 30, 20, 10, 5, 2, 1, 0.5]:
    comb = base | (rule >= thr)
    cP, cR = pr(comb)
    rows.append((thr, round(cR, 4), round(cR - bR, 4), round(cP, 4), round(cP - bP, 4)))
print(pd.DataFrame(rows, columns=["RULE_THR", "comb_R", "dR(+rule)", "comb_P", "dP"]).to_string(index=False))

TEST | total_fraud=16,384 | baseline k=0.02 (圈 20,328 人)
baseline 原来:  recall=0.7898   precision=0.6366

baseline OR (Combined_score_with_uncertain >= 阈值):
 RULE_THR  comb_R  dR(+rule)  comb_P      dP
     83.5  0.7898        0.0  0.6365 -0.0000
     50.0  0.7898        0.0  0.6319 -0.0047
     30.0  0.7898        0.0  0.5678 -0.0688
     20.0  0.7898        0.0  0.4553 -0.1813
     10.0  0.7898        0.0  0.3302 -0.3063
      5.0  0.7898        0.0  0.1863 -0.4503
      2.0  0.7898        0.0  0.1364 -0.5002
      1.0  0.7898        0.0  0.0859 -0.5506
      0.5  0.7898        0.0  0.0859 -0.5506

Connection: project.spark.compatibility | Run start time: 2026-06-09 17:58:28.983261 | Run duration : 0:00:14.212672s.


In [41]:
%%pyspark project.spark.compatibility
import numpy as np, pandas as pd, math

RULE_COL   = "Combined_score_with_uncertain"
BASELINE_K = 0.02

p    = test.select("fraud_type", PRED_COL, RULE_COL).toPandas()
y    = (p["fraud_type"] != "non_fraud").astype(int).values
pred = p[PRED_COL].astype(float).values
rule = p[RULE_COL].astype(float).values
N    = len(y)
rank = pd.Series(pred).rank(method="first", ascending=False).values
base = rank <= math.ceil(N * BASELINE_K)

base_fraud  = int((base & (y == 1)).sum())
base_normal = int((base & (y == 0)).sum())
total_fraud = int((y == 1).sum())
print(f"baseline caught {int(base.sum()):,} people = fraud {base_fraud:,} + normal {base_normal:,}")
print(f"  fraud missed by baseline = {total_fraud - base_fraud:,}\n")

print(f"rule_based_layer extra catches beyond baseline ({RULE_COL} >= threshold):")
rows = []
for thr in [83.5, 50, 30, 20, 10, 5, 2, 1, 0.5]:
    extra = (~base) & (rule >= thr)
    ef = int((extra & (y == 1)).sum())          # extra fraud caught
    en = int((extra & (y == 0)).sum())          # extra normal caught
    rows.append((thr, ef, en, base_fraud + ef, base_normal + en))
print(pd.DataFrame(rows, columns=["RULE_THR", "extra_fraud", "extra_normal",
                                  "total_fraud", "total_normal"]).to_string(index=False))


baseline 圈了 20,328 人 = fraud 12,940 + normal 7,388
  baseline 漏掉的 fraud = 3,444

rule_based_layer 在 baseline 之外多抓的（Combined_score_with_uncertain >= 阈值）:
 RULE_THR  多抓_fraud  多抓_normal  总_fraud  总_normal
     83.5         0          1    12940      7389
     50.0         0        151    12940      7539
     30.0         0       2462    12940      9850
     20.0         0       8093    12940     15481
     10.0         0      18857    12940     26245
      5.0         0      49143    12940     56531
      2.0         0      74560    12940     81948
      1.0         0     130253    12940    137641
      0.5         0     130253    12940    137641

Connection: project.spark.compatibility | Run start time: 2026-06-09 18:02:51.483410 | Run duration : 0:00:14.220777s.


In [44]:
%%pyspark project.spark.compatibility
from pyspark.sql import functions as F
import pandas as pd

PRED = "option5_reservoir_sampled_imf_anomaly_score"

def show_dist(df, name):
    d = df.withColumn("y", F.when(F.col("fraud_type") == "non_fraud", "normal").otherwise("fraud"))
    print(f"\n===== {name} =====")
    (d.groupBy("y").agg(
        F.count("*").alias("rows"),
        F.countDistinct(PRED).alias("distinct"),
        F.round(F.min(PRED), 3).alias("min"),
        F.round(F.max(PRED), 3).alias("max"),
        F.round(F.avg(PRED), 3).alias("avg")).show())
    b = (d.withColumn("bin", F.round(F.floor(F.col(PRED) / 0.05) * 0.05, 2))
          .groupBy("bin").pivot("y", ["fraud", "normal"]).count()
          .toPandas().set_index("bin").fillna(0).astype(int).sort_index())
    print("counts bucketed by 0.05:")
    print(b.to_string())

show_dist(validation, "VALIDATION")
show_dist(test, "TEST")



===== VALIDATION =====
+------+-------+--------+---+---+-----+
|     y|   rows|distinct|min|max|  avg|
+------+-------+--------+---+---+-----+
|normal|1000000|       1|0.5|0.5|  0.5|
| fraud|   1588|     669|0.0|1.0|0.661|
+------+-------+--------+---+---+-----+

0.05 分桶计数:
      fraud   normal
bin                 
0.00    166        0
0.05      5        0
0.10      7        0
0.15      9        0
0.20     55        0
0.25     95        0
0.30     25        0
0.35      6        0
0.40     53        0
0.45     64        0
0.50    102  1000000
0.55     49        0
0.60     49        0
0.65     60        0
0.70     60        0
0.75     55        0
0.80     51        0
0.85     44        0
0.90     67        0
0.95     67        0
1.00    499        0

===== TEST =====
+------+-------+--------+---+---+-----+
|     y|   rows|distinct|min|max|  avg|
+------+-------+--------+---+---+-----+
|normal|1000000|       1|0.0|0.0|  0.0|
| fraud|  16384|    4798|0.0|1.0|0.599|
+------+-------+-------

In [45]:
%%pyspark project.spark.compatibility
from pyspark.sql import functions as F
import pandas as pd

PRED = "option5_reservoir_sampled_imf_anomaly_score"
RULE = "Combined_score_with_uncertain"

def cross(df, name):
    fr = df.filter(F.col("fraud_type") != "non_fraud")          # fraud only
    cat = (F.when(F.col(RULE) == 0, "0")
            .when(F.col(RULE) <= 2,  "(0,2]")
            .when(F.col(RULE) <= 5,  "(2,5]")
            .when(F.col(RULE) <= 10, "(5,10]")
            .when(F.col(RULE) <= 20, "(10,20]")
            .otherwise("20+"))
    d = (fr.withColumn("anom_bin", F.round(F.floor(F.col(PRED) / 0.05) * 0.05, 2))  # 1st bucketing: anomaly
           .withColumn("score_cat", cat))                                            # 2nd bucketing: combined_score
    cols = ["0", "(0,2]", "(2,5]", "(5,10]", "(10,20]", "20+"]
    p = (d.groupBy("anom_bin").pivot("score_cat", cols).count()
           .toPandas().set_index("anom_bin").fillna(0).astype(int).sort_index())
    p["total"] = p.sum(axis=1)
    print(f"\n===== {name} | fraud: anomaly bucket (rows) x Combined_score bucket (cols) -> count =====")
    print(p.to_string())

cross(validation, "VALIDATION")
cross(test, "TEST")



===== VALIDATION | fraud：anomaly桶(行) × Combined_score桶(列) → 人数 =====
            0  (0,2]  (2,5]  (5,10]  (10,20]  20+   合计
anom_bin                                              
0.00      166      0      0       0        0    0  166
0.05        5      0      0       0        0    0    5
0.10        7      0      0       0        0    0    7
0.15        9      0      0       0        0    0    9
0.20       55      0      0       0        0    0   55
0.25       95      0      0       0        0    0   95
0.30       25      0      0       0        0    0   25
0.35        6      0      0       0        0    0    6
0.40       53      0      0       0        0    0   53
0.45       64      0      0       0        0    0   64
0.50      102      0      0       0        0    0  102
0.55       49      0      0       0        0    0   49
0.60       49      0      0       0        0    0   49
0.65       60      0      0       0        0    0   60
0.70       60      0      0       0        0    0 

In [ ]:
%%pyspark project.spark.compatibility



In [ ]:
%%pyspark project.spark.compatibility



In [ ]:
%%pyspark project.spark.compatibility



In [ ]:
%%pyspark project.spark.compatibility



In [ ]:
%%pyspark project.spark.compatibility



In [14]:
%%pyspark project.spark.compatibility
# import matplotlib.pyplot as plt
# from io import BytesIO
# import boto3

# best = best_p if best_p is not None else best_r      # to plot the preserve-recall version: best = best_r
# assert best is not None, "neither objective selected a rule, adjust MIN_PRECISION / MIN_RECALL first"

# # baseline net-size curve (sorted by PRED_COL)
# k_grid    = np.round(np.arange(0.002, 0.0501, 0.002), 4)
# rank_full = pd.Series(pdf[PRED_COL].values).rank(method="first", ascending=False).values
# nz        = pdf[PRED_COL].values != 0
# prec, rec, fnr, fpr = [], [], [], []
# for kk in k_grid:
#     kc = math.ceil(N * kk); innet = rank_full <= kc
#     h = int((innet & nz & (lab == 1)).sum()); fnn = int((innet & ~nz & (lab == 1)).sum())
#     fpp = int((innet & nz & (lab == 0)).sum())
#     prec.append(min(h / kc, 1.0)); rec.append(h / max(total_fraud, 1))
#     fnr.append(fnn / max(h + fnn, 1)); fpr.append(fpp / max(fpp + total_norm, 1))

# R_comb, P_comb = best["comb_R"], best["comb_P"]
# rule_lbl = f"{best['rule']} >= {best['threshold']:g}"
# C_FN, C_FP, C_PREC, C_REC, SAKURA = "#1f3b5c", "#e8743b", "#2e7d32", "#4a90d9", "#FA5A94"
# fig, ax = plt.subplots(figsize=(11, 6))
# ax.plot(k_grid, fnr, C_FN, lw=2.0); ax.plot(k_grid, fpr, C_FP, lw=2.0)
# ax.plot(k_grid, prec, C_PREC, lw=2.6); ax.plot(k_grid, rec, C_REC, lw=2.6)
# ax.axvline(BASELINE_K, color="#e53935", lw=2)
# ax.axhline(R_comb, color=SAKURA, lw=2, ls="--")
# ax.annotate("", xy=(BASELINE_K, R_comb), xytext=(BASELINE_K, R_base),
#             arrowprops=dict(arrowstyle="->", color=SAKURA, lw=2))
# ax.scatter([BASELINE_K], [R_comb], color=SAKURA, zorder=5, s=45)
# box  = dict(boxstyle="round,pad=0.3", fc="white",  ec="#cccccc")
# sbox = dict(boxstyle="round,pad=0.3", fc="#ffe5ec", ec=SAKURA)
# ax.text(BASELINE_K, 1.035, f"baseline net = {BASELINE_K:.4g}", color="#c62828", fontsize=10, fontweight="bold", ha="center")
# ax.annotate(f"baseline: P={P_base:.1%}  R={R_base:.1%}", (BASELINE_K, R_base),
#             xytext=(BASELINE_K + 0.002, R_base - 0.13), fontsize=10, bbox=box)
# ax.annotate(f"+ rule-based: R={R_comb:.1%} (P={P_comb:.1%})" + chr(10) + rule_lbl, (BASELINE_K, R_comb),
#             xytext=(BASELINE_K + 0.002, R_comb + 0.04), fontsize=10, bbox=sbox)
# ax.set_xlim(k_grid.min(), k_grid.max()); ax.set_ylim(0, 1.07)
# ax.set_xticks(k_grid); ax.set_xticklabels([f"{x:g}" for x in k_grid], fontsize=8, rotation=90)
# ax.grid(axis="y", alpha=0.3); ax.set_axisbelow(True)
# for sp in ("top", "right"): ax.spines[sp].set_visible(False)
# ax.legend(list(ax.get_lines()[:4]) + [plt.Line2D([], [], color=SAKURA, ls="--", lw=2)],
#           ["_fn", "_fp", "_precision", "_recall", "+ rule-based recall"], loc="center left", bbox_to_anchor=(1.0, 0.5), frameon=False)
# ax.set_title("IMF baseline (model score top-k)  +  Extra rule-based", fontsize=12, fontweight="bold")
# fig.tight_layout()
# buf = BytesIO(); fig.savefig(buf, format="png", dpi=150, bbox_inches="tight"); buf.seek(0)
# boto3.client("s3").put_object(
#     Bucket="amazon-sagemaker-004772120527-ca-central-1-86674be5620e",
#     Key="dzd-6m7qoryrgb21my/ae2xwmv6edkt56/shared/SherryShi/EDA/rule_validation_split/combined/plots/baseline_plus_rule.png",
#     Body=buf.getvalue(), ContentType="image/png")
# plt.show(); print("saved baseline_plus_rule.png")


Connection: project.spark.compatibility | Run start time: 2026-06-08 13:26:12.768840 | Run duration : 0:00:00.032480s.
